# Phase 5 on Kaggle: retrain SSSD-ECG (paper recipe) on our train split

Use this only if `python -m full_ecg_diffusion.probe` says the laptop is too slow or out of memory.
Prepared, **not run**. Nothing here touches val/test data: the bundle holds our train split only.

**Before you start (on your machine):**
1. Phases 1, 2, 3 and 4a passed (phase 4a confirms the model's channel order).
2. Build the bundle: `python -m full_ecg_diffusion.train --export-bundle generator_comparison/sssd_full/bundle`
3. Upload that folder as a **private** Kaggle dataset (e.g. `ecg-train-bundle`): `x.npy`, `labels.npy`, `ecg_ids.npy`, `y4.npy`, `bundle.json`.
4. Notebook settings: GPU (T4 or P100), Internet on (to clone the repo; or upload the repo as a second dataset).

Kaggle sessions stop after about 12 h. Checkpoints are written every 4,000 iterations. To continue, save `/kaggle/working/ours_train` as a dataset version and add it as an input, then set `RESUME` below.

In [ ]:
!pip -q install einops opt_einsum
!git clone -q https://github.com/shivankmishra144droid/Quality-Aware-Synthetic-Data-Augmentation-for-ECG-Classification.git repo
%cd repo
!git checkout -q claude/nifty-bell-m5a6en

In [ ]:
BUNDLE = "/kaggle/input/ecg-train-bundle"      # your private dataset
OUT = "/kaggle/working/ours_train"
RESUME = ""                                     # e.g. "/kaggle/input/ours-train-ckpt/ours_48000.pt"
import torch; print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
# Measure first: picks micro-batch / checkpointing for this GPU and projects the 100k-iteration time.
!python -m full_ecg_diffusion.probe --out-root /kaggle/working --steps 3

In [ ]:
# Set --micro-batch (and --grad-checkpointing) from the probe recommendation above.
resume = f"--resume {RESUME}" if RESUME else ""
!python -m full_ecg_diffusion.train --bundle {BUNDLE} --out {OUT} --preset paper --micro-batch 6 {resume}

Afterwards, download `ours_last.pt` (or the newest `ours_<iter>.pt`) into `generator_comparison/sssd_full/ours_train/` on your machine and continue with phase 6 (`sample`, then `evaluate`).